<a href="https://colab.research.google.com/github/Kwasi-Dankwa/FinanceBrokerage_Project_RAG/blob/main/AI%26DS_Project_3_Full_Code_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement

### Business Context

GlobalEdge Brokerage is a mid-sized brokerage firm operating across multiple countries, serving thousands of retail and institutional clients through a network of equity brokers. Brokers handle investment recommendations, portfolio reviews, and market advisory discussions across equity, forex, commodity, crypto, and international stock markets. Each broker manages hundreds of clients and is expected to stay updated with overnight market developments before the market opens.

Every morning, brokers must review large volumes of financial news, stock-price movements, earnings discussions, analyst commentary, and SEC regulatory filings within a very limited preparation window. In practice, brokers can only read a small portion of the available information before their first client calls begin. As a result, many important signals, disclosures, and market events remain unnoticed.

This creates an intelligence gap during client interactions. Brokers often rely on partial information, memory, or fragmented market sources while answering client questions. Important disclosures buried inside long 10-K and 10-Q filings are difficult to review manually, and brokers may struggle to provide evidence-backed responses when clients ask for justification or supporting references. This increases both compliance risk and trust-related challenges.

To solve this problem, GlobalEdge wants to build a financial intelligence assistant that allows brokers to ask natural-language questions and receive grounded answers backed by real financial data, news articles, and regulatory filings. The system should reduce information overload, improve market coverage, and help brokers make faster and more confident advisory decisions without adding technical complexity to their workflow.

### Objective

This project proposes a proof-of-concept Retrieval-Augmented Generation (RAG) financial intelligence system that combines financial news, stock-price data, and SEC filings into a searchable intelligence layer. The system uses semantic retrieval with a local Chroma vector database to fetch relevant financial context and generate grounded answers for broker questions using large language models. Brokers can ask plain-English questions such as market sentiment analysis, company-risk queries, filing-related questions, or cross-market comparisons and receive evidence-backed responses.

To improve the quality and reliability of generated answers, the system introduces a DeepEval-based prompt optimization workflow using GEPA (Genetic-Pareto Prompt Optimization). Instead of manually refining prompts, the workflow uses benchmark financial question-answer examples and evaluation metrics such as faithfulness, groundedness, relevance, and actionability to optimize the final answering prompt.

The project also evaluates multiple RAG configurations by tuning retrieval and generation parameters such as chunk size, chunk overlap, number of retrieved chunks, temperature, top-p, and maximum token limits. Each configuration is evaluated using DeepEval metrics to identify the most effective combination for grounded financial reasoning and broker-style question answering.

The final system uses the optimized prompt together with the best-performing RAG configuration to generate accurate, grounded, and production-style financial intelligence responses for broker workflows.

### Data Description

The system uses three financial intelligence datasets collected through a custom data ingestion pipeline and stored locally for the RAG workflow.
1. Global Financial News (global_news.csv)
Contains financial news articles with titles, article content, publication dates, URLs, and source metadata. The dataset covers company announcements, market developments, macroeconomic events, sector movements, and sentiment-related signals.
2. Global Stock Prices (stock_price_details.csv)
Contains historical stock-price data across global equity markets, crypto markets, and forex markets. Each record includes ticker symbols, company names, open/high/low/close prices, trading volume, and timestamps for market analysis and trend evaluation.
3. SEC Regulatory Filings (sec_filings_10k.txt)
Contains regulatory filing documents including 10-K annual reports, 10-Q quarterly reports, and other SEC disclosures. These filings include financial statements, governance information, risk disclosures, operational commentary, and compliance-related information used for grounded financial reasoning.

## Installing and Importing Necessary Libraries and Dependencies

In [3]:
# Installing the necessary libraries with compatible versions
%pip install -qU \
    "chromadb==1.5.9" \
    "langchain-community" \
    "langchain-chroma" \
    "langchain-openai==1.2.1" \
    "langchain-text-splitters==1.1.2" \
    "pandas==3.0.3" \
    "numpy==2.4.4" \
    "scikit-learn>=1.4.0" \
    "python-dotenv>=1.0.1" \
    "tqdm>=4.66.0"\
    "deepeval==4.2.2"

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for VSCode), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [6]:
pip install --upgrade langchain langchain-core langchain-community langchain-text-splitters langchain-chroma langchain-openai deepeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.7/574.7 kB 15.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 39.1 MB/s  0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.12
    Uninstalling langchain-core-1.6.12:
      Successfully uninstalled langchain-core-1.6.12
  Attempting uninstall: deepeval
    Found existing installation: deepeval 4.2.2
    Uninstalling deepeval-4.2.2:
      Successfully uninstalled deepeval-4.2.2
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 1.1.2
    Uninstalling langchain-text-splitters-1.1.2:
      Successfully uninstalled langchain-text-splitters-1.1.2
  Attempting uninstall: langchain-openai
    Found existing installation: langchain-openai 1.2.1
    Uninstalling langchain-openai-1.2.1:
      Successfully uninstalled langchain-openai-1.2.1
  Attempting uninstall: langchain
    Found existing installation: langchain 1.4.6
  

In [13]:
# Importing the necessary libraries

# Standard Python libraries for file handling, text processing, JSON handling,
# randomness, timing, and basic utilities.
import os
import re
import json
import random
import time
from pathlib import Path
from collections import Counter
import pandas as pd

# DeepEval libraries for prompt optimization and evaluation.
from deepeval.prompt import Prompt
from deepeval.dataset import Golden
from deepeval.metrics import GEval
from deepeval.optimizer import PromptOptimizer
from deepeval.optimizer.algorithms import GEPA
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.models import OpenAIModel
from deepeval.optimizer.policies import TieBreaker
from deepeval.metrics import utils as deepeval_metric_utils
from deepeval.optimizer.configs import AsyncConfig

# LangChain libraries for document loading, text splitting, embeddings,
# vector storage, and LLM interaction.
from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

## Environment Setup and Project Initialization

In [14]:
# Write your code here
# Load environment variables (API keys, Base URL, Model Names)
from dotenv import load_dotenv
load_dotenv()

# Verify and set model configuration constants
EVAL_MODEL = os.getenv("EVAL_MODEL", "gpt-4o")
ANSWER_MODEL = os.getenv("ANSWER_MODEL", "gpt-4o-mini")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")

print("Environment setup completed successfully.")

Environment setup completed successfully.


Text-embedding-3-small significantly outperforms OpenAI's older generation model (text-embedding-ada-002) on standard embedding benchmarks like MTEB (Massive Text Embedding Benchmark). For a financial RAG system, this means the vector search is more accurate at capturing semantic nuances—crucial when brokers are looking up specific regulatory disclosures, risk factors, or market sentiment across dense text.

# Section 1: Baseline Retrieval-Augmented Generation (RAG) Pipeline

## 1.1 Load the CSV files with `CSVLoader`

In [15]:
# Write your code here
# Load global financial news dataset
news_loader = CSVLoader(file_path="global_news.csv")
news_docs = news_loader.load()

# Load stock price details dataset
stock_loader = CSVLoader(file_path="stock_price_details.csv")
stock_docs = stock_loader.load()

print(f"Loaded {len(news_docs)} news documents.")
print(f"Loaded {len(stock_docs)} stock price documents.")

Loaded 760 news documents.
Loaded 2986 stock price documents.


CSVLoader is needed to convert tabular spreadsheet rows (like your CSV files) into text-based Document objects that vector databases and embedding models can read, embed, and search.

## 1.2 Load the SEC filings text and split it into chunks


In [16]:
# Write your code here
# Load SEC filings text file
sec_loader = TextLoader("sec_filings_10k.txt", encoding="utf-8")
sec_docs = sec_loader.load()

# Initialize RecursiveCharacterTextSplitter with baseline chunk settings
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

# Split SEC filings into document chunks
sec_chunks = text_splitter.split_documents(sec_docs)

# Combine all loaded documents into a single collection for the vector store
all_documents = news_docs + stock_docs + sec_chunks

print(f"Loaded and split SEC filings into {len(sec_chunks)} chunks.")
print(f"Total documents prepared for indexing: {len(all_documents)}")

Loaded and split SEC filings into 10065 chunks.
Total documents prepared for indexing: 13811


Text splitter to break it down into manageable chunks (1,000 characters with a 200-character overlap) for precise semantic retrieval.

## 1.3 Build a local Chroma vector store

In [18]:
import os
from dotenv import load_dotenv

# Force load the .env file
load_dotenv()

# Check if the key is detected (this should print your key, NOT None)
print("API Key loaded:", os.getenv("OPENAI_API_KEY") is not None)

# If it prints 'False', set it directly right here:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "your_actual_openai_api_key_here"

API Key loaded: True


In [19]:
# Write your code here
# Instantiate OpenAI Embeddings model
embedding_function = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

# Build and persist local Chroma vector database
vectorstore = Chroma.from_documents(
    documents=all_documents,
    embedding=embedding_function,
    collection_name="financial_intelligence",
    persist_directory="./chroma_db",
)

# Create baseline retriever (fetching top k=4 chunks)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("Local Chroma vector store built and persisted successfully.")

Local Chroma vector store built and persisted successfully.


Initializes the OpenAI embedding model to transform all prepared document chunks into numerical vector representations and stores them within a local Chroma vector database saved to ./chroma_db. The point of the Chroma vector database is to function as the semantic retrieval engine for the RAG system; rather than relying on brittle keyword matching, it allows the application to mathematically search and pull the most relevant financial news, stock prices, or SEC filing segments (fetching the top $k=4$ chunks) needed to accurately ground the AI's response.  

## 1.4 Retrieve Context and Generate Answers using the Baseline RAG Pipeline

In [20]:
# Write your code here
# Define baseline system and user prompts
BASELINE_SYSTEM_PROMPT = (
    "You are a helpful financial intelligence assistant for brokers at GlobalEdge Brokerage. "
    "Answer the broker's question clearly, professionally, and strictly using the provided context."
)

BASELINE_USER_PROMPT = (
    "Context:\n{context}\n\nQuestion:\n{question}\n\nAnswer:"
)

# Instantiate the baseline LLM
baseline_llm = ChatOpenAI(
    model=ANSWER_MODEL,
    temperature=0.0,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

def baseline_rag_pipeline(question: str) -> dict:
    """
    Retrieves context using Chroma vectorstore and generates an answer using baseline LLM.
    Returns a dictionary with query, context, and actual answer.
    """
    # 1. Retrieve relevant document chunks
    retrieved_docs = retriever.invoke(question)
    context_str = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # 2. Format user prompt with context and question
    formatted_user_prompt = BASELINE_USER_PROMPT.format(
        context=context_str,
        question=question
    )

    # 3. Call LLM
    response = baseline_llm.invoke([
        SystemMessage(content=BASELINE_SYSTEM_PROMPT),
        HumanMessage(content=formatted_user_prompt)
    ])

    return {
        "question": question,
        "context": [context_str],
        "actual_output": response.content
    }

# Quick sanity check test
sample_res = baseline_rag_pipeline("What are the key financial risk factors disclosed in 10-K filings?")
print("Baseline RAG Sample Output:\n", sample_res["actual_output"])

Baseline RAG Sample Output:
 The key financial risk factors disclosed in the 10-K filings include:

1. **Macroeconomic and Industry Risks**: The Company's operations and performance are significantly influenced by global and regional economic conditions. Adverse economic conditions can materially affect the Company's business, results of operations, and financial condition.

2. **International Operations**: A majority of the Company's total net sales come from international operations, making it vulnerable to risks associated with foreign markets.

3. **Complex Global Supply Chain**: The Company's large and complex global supply chain, which includes a majority of supplier facilities for manufacturing and assembly, can be affected by various risks that may disrupt operations.

4. **Material Changes in Risk Factors**: There have been no material changes to the Company's risk factors since the 2024 Form 10-K, except for those disclosed in the first and second quarter 2025 Form 10-Q filin

Sets the operational behavior for the model, instructing it to act as a professional financial assistant for brokers and to answer questions strictly using the provided context.

I'm setting temp to 0 to ensure the outputs remain deterministic, consistent and factual

## 1.5 Load the gold benchmark dataset and evaluate the baseline prompt


In [22]:
import pandas as pd
from deepeval.dataset import Golden

# Load benchmark evaluation dataset from CSV
df_goldens = pd.read_csv("golden_benchmark_dataset.csv")

# Convert dataframe rows to DeepEval Golden objects using the correct column names
deepeval_goldens = []
for _, row in df_goldens.iterrows():
    deepeval_goldens.append(
        Golden(
            input=row["question"],         # Mapped from the 'question' column[cite: 2]
            expected_output=row["response"], # Mapped from the 'response' column[cite: 2]
            context=[]                     # Context will be dynamically retrieved by your RAG retriever during evaluation
        )
    )

# Split into train/validation goldens for prompt optimization and baseline testing
train_goldens = deepeval_goldens[: int(len(deepeval_goldens) * 0.7)]
test_goldens = deepeval_goldens[int(len(deepeval_goldens) * 0.7):]

print(f"Loaded {len(deepeval_goldens)} benchmark goldens total from CSV.")
print(f"Train goldens: {len(train_goldens)} | Test goldens: {len(test_goldens)}")

Loaded 20 benchmark goldens total from CSV.
Train goldens: 14 | Test goldens: 6


## 1.6 Define the evaluation metrics

In [23]:
# Define the evaluation criteria.

RELEVANCE_CRITERIA = (
    "Evaluate whether the actual output directly answers the question provided in the input "
    "and matches the expected output in terms of core factual information, financial metrics, and company context."
)

COMPLETENESS_CRITERIA = (
    "Evaluate whether the actual output thoroughly covers all aspects and key details requested in the input question "
    "using the provided context without omitting essential financial signals."
)

FAITHFULNESS_CRITERIA = (
    "Evaluate whether every statement and claim in the actual output is strictly supported by and grounded in "
    "the provided context, containing no hallucinations or unsourced assumptions."
)

In [ ]:
# Evaluation model used by DeepEval metrics.
eval_gpt_model = OpenAIModel(
    model=EVAL_MODEL,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

# 1. Relevance:
# Checks whether the generated answer aligns with the expected benchmark answer.
relevance_metric = GEval(
    name="Relevance",
    criteria=RELEVANCE_CRITERIA,
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    model=eval_gpt_model,
    threshold=_____,    # complete the code to define the minimum score  threshold required for the evaluation to pass
    async_mode=False,
)

# 2. Faithfulness:
# Checks whether the answer stays grounded in the retrieved context.
faithfulness_metric = GEval(
    name="Faithfulness",
    criteria=FAITHFULNESS_CRITERIA,
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.CONTEXT,
    ],
    model=eval_gpt_model,
    threshold=_____,     # complete the code to define the minimum score  threshold required for the evaluation to pass
    async_mode=False,

)

# 3. Completeness:
# Checks whether the answer fully covers the important aspects of the question.
completeness_metric = GEval(
    name="Completeness",
    criteria=COMPLETENESS_CRITERIA,
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.CONTEXT,
    ],
    model=eval_gpt_model,
    threshold=_____,    # complete the code to define the minimum score  threshold required for the evaluation to pass
    async_mode=False,
)


# Final metric list used across:
# - baseline evaluation
# - GEPA optimization
# - RAG configuration tuning

# uncomment the metrics to use for prompt optimization
DEEPEVAL_METRICS = [
    # relevance_metric,
    # faithfulness_metric,
    # completeness_metric,
]

In [24]:
# Write your code here
# Evaluation model used by DeepEval metrics.
eval_gpt_model = OpenAIModel(
    model=EVAL_MODEL,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

# 1. Relevance:
# Checks whether the generated answer aligns with the expected benchmark answer.
relevance_metric = GEval(
    name="Relevance",
    criteria=RELEVANCE_CRITERIA,
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    model=eval_gpt_model,
    threshold=0.7,    # minimum score threshold required to pass
    async_mode=False,
)

# 2. Faithfulness:
# Checks whether the answer stays grounded in the retrieved context.
faithfulness_metric = GEval(
    name="Faithfulness",
    criteria=FAITHFULNESS_CRITERIA,
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.CONTEXT,
    ],
    model=eval_gpt_model,
    threshold=0.8,     # minimum score threshold required to pass
    async_mode=False,
)

# 3. Completeness:
# Checks whether the answer fully covers the important aspects of the question.
completeness_metric = GEval(
    name="Completeness",
    criteria=COMPLETENESS_CRITERIA,
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.CONTEXT,
    ],
    model=eval_gpt_model,
    threshold=0.7,    # minimum score threshold required to pass
    async_mode=False,
)

# Final metric list used across baseline evaluation, GEPA optimization, and tuning
DEEPEVAL_METRICS = [
    relevance_metric,
    faithfulness_metric,
    completeness_metric,
]

## 1.7 Evaluate the baseline RAG on the golden examples dataset

In [26]:
# Run baseline evaluation across test goldens
baseline_test_cases = []

for golden in test_goldens:
    rag_out = baseline_rag_pipeline(golden.input)

    test_case = LLMTestCase(
        input=golden.input,
        actual_output=rag_out["actual_output"],
        expected_output=golden.expected_output,
        context=rag_out["context"]  # Changed from retrieval_context to context
    )

    # Measure metrics for each baseline test case
    for metric in DEEPEVAL_METRICS:
        metric.measure(test_case)

    baseline_test_cases.append(test_case)

# Extract scores summary
baseline_scores = {
    "Relevance": [m.score for tc in baseline_test_cases for m in DEEPEVAL_METRICS if m.name == "Relevance"],
    "Faithfulness": [m.score for tc in baseline_test_cases for m in DEEPEVAL_METRICS if m.name == "Faithfulness"],
    "Completeness": [m.score for tc in baseline_test_cases for m in DEEPEVAL_METRICS if m.name == "Completeness"],
}

baseline_avg_scores = {k: sum(v)/len(v) if v else 0.0 for k, v in baseline_scores.items()}
print("Baseline Evaluation Complete!")
print("Baseline Average Metric Scores:", baseline_avg_scores)

Output()

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

Baseline Evaluation Complete!
Baseline Average Metric Scores: {'Relevance': 0.16684648169977043, 'Faithfulness': 0.9999999999999997, 'Completeness': 0.9444733362325698}


# Section 2: DeepEval Prompt Optimization Layer

## 2.1 Define the prompt template and model callback


In [71]:
import os
import time
import openai
from dotenv import load_dotenv

# 1. Establish and secure environment variables
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL")

if not api_key:
    raise ValueError("OPENAI_API_KEY is not set. Please check your .env/environment configuration.")

os.environ["OPENAI_API_KEY"] = api_key
if base_url:
    os.environ["OPENAI_BASE_URL"] = base_url
    os.environ["OPENAI_API_BASE"] = base_url

openai.api_key = api_key
if base_url:
    openai.base_url = base_url

from deepeval.optimizer import PromptOptimizer
from deepeval.optimizer.algorithms import GEPA
from deepeval.optimizer.configs import AsyncConfig
from deepeval.optimizer.policies import TieBreaker
from deepeval.models import OpenAIModel
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams
from langchain_openai import ChatOpenAI
from deepeval.prompt import Prompt
from deepeval.dataset import Golden
from langchain_core.messages import SystemMessage, HumanMessage
from deepeval.metrics import utils as deepeval_metric_utils

# Decouple token usage accrual patch if needed
deepeval_metric_utils.accrue_token_usage = lambda metric, cost: None

# 2. Define Prompt Template and Models
PROMPT_TEMPLATE_V1 = Prompt(
    text_template=(
        BASELINE_SYSTEM_PROMPT
        + "\n\n"
        + BASELINE_USER_PROMPT
            .replace("{question}", "{input}")
            .replace("{context}", "{context}")
    )
)

optim_model = OpenAIModel(
    model=ANSWER_MODEL,
    temperature=0.3,
    api_key=api_key,
    base_url=base_url
)

eval_gpt_model = OpenAIModel(
    model=EVAL_MODEL,
    api_key=api_key,
    base_url=base_url
)

# 3. Define Model Callback with Local Client Instantiation
def model_callback(prompt: Prompt, golden: Golden) -> str:
    # Re-enforce environment variables inside callback scope
    os.environ["OPENAI_API_KEY"] = api_key
    if base_url:
        os.environ["OPENAI_BASE_URL"] = base_url
        os.environ["OPENAI_API_BASE"] = base_url

    # Instantiate ChatOpenAI locally here so the API key is never dropped
    local_answer_llm = ChatOpenAI(
        model=ANSWER_MODEL,
        temperature=0.0,
        openai_api_key=api_key,
        openai_api_base=base_url
    )

    interpolated = prompt.interpolate(
        input=golden.input,
        context=golden.context[0] if golden.context else "",
    )

    response = local_answer_llm.invoke([
        SystemMessage(content=(
            "You are a financial intelligence assistant for brokers. "
            "Follow the user prompt exactly and answer only from the provided context."
        )),
        HumanMessage(content=interpolated),
    ])

    return response.content

# 4. Initialize Evaluation Metrics
relevance_metric = GEval(
    name="Relevance",
    criteria=RELEVANCE_CRITERIA,
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
    model=eval_gpt_model,
    threshold=0.7,
    async_mode=False,
)

faithfulness_metric = GEval(
    name="Faithfulness",
    criteria=FAITHFULNESS_CRITERIA,
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT],
    model=eval_gpt_model,
    threshold=0.8,
    async_mode=False,
)

completeness_metric = GEval(
    name="Completeness",
    criteria=COMPLETENESS_CRITERIA,
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT],
    model=eval_gpt_model,
    threshold=0.7,
    async_mode=False,
)

DEEPEVAL_METRICS = [relevance_metric, faithfulness_metric, completeness_metric]

# 5. Run GEPA Optimization
start_time = time.time()

gepa_optimizer = PromptOptimizer(
    algorithm=GEPA(
        iterations=3,
        pareto_size=3,
        minibatch_size=5,
        random_seed=42,
        tie_breaker=TieBreaker.PREFER_CHILD,
    ),
    model_callback=model_callback,
    metrics=DEEPEVAL_METRICS,
    optimizer_model=optim_model,
    async_config=AsyncConfig(run_async=False),
)

print("Starting prompt optimization. This may take a few minutes...")
gepa_prompt = gepa_optimizer.optimize(
    prompt=PROMPT_TEMPLATE_V1,
    goldens=train_goldens,
)

end_time = time.time()
print(f"\nOptimization Execution Time: {end_time - start_time:.4f} seconds")
print("\nGEPA Optimized Prompt Template:")
print(gepa_prompt.text_template)

Output()

Starting prompt optimization. This may take a few minutes...


[GEPA] (iterations=3) • error AuthenticationError: Error code: 401 - {'error': {'message': "You didn't provide an 
API key. You need to provide your API key in an Authorization header using Bearer auth (i.e. Authorization: Bearer 
YOUR_KEY), or as the password field (with blank username) if you're accessing the API from your browser and are 
prompted for a username and password. You can obtain an API key from 
https://platform.openai.com/account/api-keys.", 'type': 'invalid_request_error', 'param': None, 'code': None}} • 
halted at iteration 1

AuthenticationError: Error code: 401 - {'error': {'message': "You didn't provide an API key. You need to provide your API key in an Authorization header using Bearer auth (i.e. Authorization: Bearer YOUR_KEY), or as the password field (with blank username) if you're accessing the API from your browser and are prompted for a username and password. You can obtain an API key from https://platform.openai.com/account/api-keys.", 'type': 'invalid_request_error', 'param': None, 'code': None}}

## 2.2 Optimize the prompt with GEPA

## 2.3 Evaluate the optimized prompt on the benchmark dataset

In [72]:
# Write your code here
from deepeval.prompt import Prompt

# Define a robust, professional prompt manually to replace the automated GEPA output
OPTIMIZED_FINANCIAL_PROMPT = (
    "You are an expert financial intelligence assistant for brokers at GlobalEdge Brokerage. "
    "Your task is to provide accurate, professional, and evidence-backed answers to broker queries. "
    "Analyze the provided context thoroughly. "
    "Strictly ground your response in the provided context and avoid making unsourced claims.\n\n"
    "Context:\n{context}\n\n"
    "Question:\n{input}\n\n"
    "Answer:"
)

# Wrap it in DeepEval's Prompt object so it stays compatible with your evaluation workflow
gepa_prompt = Prompt(text_template=OPTIMIZED_FINANCIAL_PROMPT)

print("Prompt successfully set manually. You can now proceed to Section 3 and Section 4!")
print("\nTemplate Preview:\n", gepa_prompt.text_template)

Prompt successfully set manually. You can now proceed to Section 3 and Section 4!

Template Preview:
 You are an expert financial intelligence assistant for brokers at GlobalEdge Brokerage. Your task is to provide accurate, professional, and evidence-backed answers to broker queries. Analyze the provided context thoroughly. Strictly ground your response in the provided context and avoid making unsourced claims.

Context:
{context}

Question:
{input}

Answer:


## 2.4 Compare optimized prompt with the baseline prompt


In [73]:
# Write your code here
# Cell 2.4: Compare Baseline and Optimized Prompts

print("=" * 60)
print("BASELINE PROMPT STRUCTURE")
print("=" * 60)
print(f"System Message:\n{BASELINE_SYSTEM_PROMPT}\n")
print(f"User Template:\n{BASELINE_USER_PROMPT}")

print("\n" + "=" * 60)
print("OPTIMIZED PROMPT STRUCTURE")
print("=" * 60)
print(f"Combined Template:\n{gepa_prompt.text_template}")
print("=" * 60)

BASELINE PROMPT STRUCTURE
System Message:
You are a helpful financial intelligence assistant for brokers at GlobalEdge Brokerage. Answer the broker's question clearly, professionally, and strictly using the provided context.

User Template:
Context:
{context}

Question:
{question}

Answer:

OPTIMIZED PROMPT STRUCTURE
Combined Template:
You are an expert financial intelligence assistant for brokers at GlobalEdge Brokerage. Your task is to provide accurate, professional, and evidence-backed answers to broker queries. Analyze the provided context thoroughly. Strictly ground your response in the provided context and avoid making unsourced claims.

Context:
{context}

Question:
{input}

Answer:


# Section 3: RAG Configuration Tuning and Evaluation


## 3.1 Define the RAG configurations

In [ ]:
# Write your code here

## 3.2 Create a runtime answering function

In [ ]:
# Write your code here

## 3.3 Evaluate one configuration on a benchmark set

In [ ]:
# Write your code here

## 3.4 Run all configurations on the test set

In [ ]:
# Write your code here

## 3.5 Compare configurations

In [ ]:
# Write your code here

## 3.6 Select the best overall approach

In [ ]:
# Write your code here

## 3.7 Save the tuning results


In [ ]:
# Write your code here

# Section 4: Final RAG Inference Pipeline with the Optimized Prompt and Best RAG Configuration

## 4.1 Define the final inference helper

In [ ]:
# Write your code here

## 4.2 Define the final test cases

In [ ]:
final_test_cases = [
    {
        "id": 1,
        "question": (
            "Compare the total revenue and net income reported by Apple, Amazon, "
            "and Alphabet in their latest 10-Q filings. Which company grew fastest "
            "year-over-year?"
        )
    },
    {
        "id": 2,
        "question": (
            "Based on this week's news sentiment around Amazon, would you classify "
            "the current signal as bullish or bearish, and does the price data support that?"
        )
    },
    {
        "id": 3,
        "question": (
            "Should I invest in Apple right now? I heard Tim Cook is stepping down — "
            "is that a red flag? How's the stock been doing, and is there anything I should "
            "be worried about with the company's financials or risks?"
        )
    },
    {
        "id": 4,
        "question": (
            "Is now a good time to buy Bitcoin? News is saying it just crossed $78k — "
            "am I too late? What's the price trend looked like over the last few months, "
            "and what's driving the current rally?"
        )
    },
    {
        "id": 5,
        "question": (
            "I want to invest in AI. Out of Google, Amazon, and Apple, which one looks "
            "like the best bet right now? How are they performing, what are they saying "
            "about their AI business, and what's the buzz around them?"
        )
    },
]

## 4.3 Run the final test cases

In [ ]:
# Write your code here

# Conclusion

## Actionable Insights:

- WRITE INSIGHTS HERE

## Recommendations:

- WRITE RECOMMENDATIONS HERE

<font size=6>Power Ahead!</font>
___